# Feature Selection

Scores every feature of the engineered dataset with two order-agnostic
methods - mutual information and RF permutation importance (stratified
5-fold, macro-F1, class-weighted for imbalance) - prunes redundant pairs
(|Spearman rho| > 0.9), rolls one-hot dummies up to their parent feature,
and writes the proposed list to `configs/config.json` as
`selected_features` (the only write this notebook makes).

DS2 uses the Feature_Engineering output; DS1 uses its cleaned csv with
in-notebook binary encoding and in-memory median imputation for scoring.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.inspection import permutation_importance
from sklearn.model_selection import StratifiedKFold

root = Path.cwd()
while not (root / "configs" / "config.json").is_file():
    if root == root.parent:
        raise FileNotFoundError("configs/config.json not found above the working directory")
    root = root.parent

CONFIG_PATH = root / "configs" / "config.json"
config = json.loads(CONFIG_PATH.read_text())
RANDOM_STATE = config["training"]["random_state"]


def dataset_file(ds_id, stage):
    """Path of one dataset file; config paths are relative to configs/."""
    ds = config["datasets"]["list"][ds_id]
    entry = ds["files"][stage]
    if isinstance(entry, dict):
        entry = entry.get("train", entry.get("data"))
    elif isinstance(entry, list):
        entry = entry[0]
    return root / "configs" / config["paths"]["datasets_dir"] / ds["folder"] / entry


def prepare(ds_id):
    """Numeric matrix + target ready for scoring (in-memory only)."""
    ds = config["datasets"]["list"][ds_id]
    if ds_id == "DS2":
        df = pd.read_csv(dataset_file(ds_id, "engineering"))
        y = df["Target"]
        X = df.drop(columns=["Target"])
    else:
        df = pd.read_csv(dataset_file(ds_id, "cleaned"), na_values=ds["na_values"] or None)
        y = df["class"].map({"notckd": 0, "ckd": 1})
        X = df.drop(columns=["class"])
        for c in X.columns:
            if pd.api.types.is_string_dtype(X[c]):
                X[c] = X[c].map({v: i for i, v in enumerate(sorted(X[c].dropna().unique()))})
    return X.fillna(X.median(numeric_only=True)), y

In [2]:
def mi_scores(X, y, discrete):
    s = mutual_info_classif(X, y, discrete_features=discrete, random_state=RANDOM_STATE)
    return pd.Series(s, index=X.columns, name="mutual_info")


def permutation_scores(X, y):
    rf = RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                random_state=RANDOM_STATE, n_jobs=-1)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    imp = np.zeros(X.shape[1])
    for train, test in skf.split(X, y):
        rf.fit(X.iloc[train], y.iloc[train])
        r = permutation_importance(rf, X.iloc[test], y.iloc[test],
                                   scoring="f1_macro", n_repeats=5,
                                   random_state=RANDOM_STATE, n_jobs=-1)
        imp += r.importances_mean
    return pd.Series(imp / skf.get_n_splits(), index=X.columns, name="perm_importance")

In [3]:
N_KEEP = {"DS2": 15, "DS1": 12}
selected = {}

for ds_id in ("DS2", "DS1"):
    X, y = prepare(ds_id)
    discrete = np.array([X[c].nunique() <= 6 for c in X.columns])
    mi = mi_scores(X, y, discrete)
    perm = permutation_scores(X, y)

    table = pd.DataFrame({"mutual_info": mi, "perm_importance": perm})
    table["mi_rank"] = mi.rank(ascending=False, method="min")
    table["perm_rank"] = perm.rank(ascending=False, method="min")
    table["avg_rank"] = (table["mi_rank"] + table["perm_rank"]) / 2
    table = table.sort_values("avg_rank")

    corr = X.corr(method="spearman").abs()
    kept = []
    for col in table.index:
        if any(corr.loc[col, k] > 0.9 for k in kept):
            continue
        kept.append(col)
    pruned = [c for c in table.index if c not in kept]

    print(f"===== {ds_id} - {config['datasets']['list'][ds_id]['name']} =====")
    print(f"scored {X.shape[1]} columns over {y.nunique()} classes; "
          f"redundancy pruning removed: {pruned or 'nothing'}\n")
    print(table.round(4).to_string())

    def parent_name(col):
        return "Physical activity level" if col.startswith("Physical activity level_") else col

    rolled = table.loc[kept].copy()
    grouped = rolled.groupby(rolled.index.map(parent_name)).agg(
        mutual_info=("mutual_info", "max"),
        perm_importance=("perm_importance", "max"),
        avg_rank=("avg_rank", "min"),
    ).sort_values("avg_rank")
    print("\nFinal ranking (dummies rolled up to their parent feature):")
    print(grouped.round(4).to_string())

    chosen = list(grouped.index[:N_KEEP[ds_id]])
    selected[ds_id] = chosen
    print(f"\nPROPOSED selected_features (top {N_KEEP[ds_id]} by average rank):")
    print(chosen)
    print()

/Users/elgazar/Desktop/RoboticsMasters/Semester2_Sep/F21DL/Project/F21DL_Group_13/.venv/lib/python3.14/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


/Users/elgazar/Desktop/RoboticsMasters/Semester2_Sep/F21DL/Project/F21DL_Group_13/.venv/lib/python3.14/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


/Users/elgazar/Desktop/RoboticsMasters/Semester2_Sep/F21DL/Project/F21DL_Group_13/.venv/lib/python3.14/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


/Users/elgazar/Desktop/RoboticsMasters/Semester2_Sep/F21DL/Project/F21DL_Group_13/.venv/lib/python3.14/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


/Users/elgazar/Desktop/RoboticsMasters/Semester2_Sep/F21DL/Project/F21DL_Group_13/.venv/lib/python3.14/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


===== DS2 - KG Kidney Disease Dataset =====
scored 46 columns over 5 classes; redundancy pruning removed: nothing

                                             mutual_info  perm_importance  mi_rank  perm_rank  avg_rank
Specific gravity of urine                         0.0065           0.0000      1.0        7.0       4.0
Packed cell volume (%)                            0.0040           0.0000      2.0        7.0       4.5
Sodium level (mEq/L)                              0.0021           0.0000      3.0        7.0       5.0
Urine protein-to-creatinine ratio                 0.0012           0.0000      5.0        7.0       6.0
Sugar in urine                                    0.0009           0.0000      6.0        7.0       6.5
Cholesterol level                                 0.0007           0.0000      7.0        7.0       7.0
Albumin in urine                                  0.0006           0.0000      8.0        7.0       7.5
Pus cells in urine                                0.0

/Users/elgazar/Desktop/RoboticsMasters/Semester2_Sep/F21DL/Project/F21DL_Group_13/.venv/lib/python3.14/site-packages/sklearn/metrics/cluster/_supervised.py:68: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and binary values for target
  warnings.warn(msg, UserWarning)


===== DS1 - UCI Chronic Kidney Disease Dataset =====
scored 24 columns over 2 classes; redundancy pruning removed: nothing

       mutual_info  perm_importance  mi_rank  perm_rank  avg_rank
sc          0.3746           0.0054      3.0        2.0       2.5
hemo        0.4446           0.0027      1.0        4.0       2.5
sg          0.3164           0.1828      5.0        1.0       3.0
al          0.2526           0.0049      7.0        3.0       5.0
htn         0.2341           0.0027      8.0        5.0       6.5
dm          0.2123           0.0016      9.0        6.0       7.5
bgr         0.1603           0.0005     12.0        7.0       9.5
pot         0.2021           0.0000     10.0       10.0      10.0
bu          0.1801           0.0000     11.0       10.0      10.5
pc          0.1023           0.0005     16.0        7.0      11.5
pe          0.1023           0.0005     16.0        7.0      11.5
bp          0.1365           0.0000     13.0       10.0      11.5
pcv         0.4144

In [4]:
for ds_id, features in selected.items():
    config["datasets"]["list"][ds_id]["selected_features"] = features
CONFIG_PATH.write_text(json.dumps(config, indent=4, ensure_ascii=False) + "\n")
print("wrote selected_features to configs/config.json for:", ", ".join(selected))

wrote selected_features to configs/config.json for: DS2, DS1
